# VetDoğum — Sentetik Veri ile Model Geliştirme

TÜBİTAK 2209-A kapsamında desteklenmeye kabul edilen bitirme projesinin deneysel model çalışması.

Notebook Google Colab üzerinde çalıştırılmak üzere hazırlanmıştır. Veri üretimi ve eğitim kodu orijinal çalışmadan korunmuştur; kayıtlı çıktılar temizlenmiştir. Tüm hücreleri çalıştırarak sonuçları yeniden üretin. Sonuçlar sentetik veri üzerindeki deneyleri ifade eder; gerçek sensör verileriyle doğrulama gelecek çalışma kapsamındadır.


In [ ]:
# ============================================================================
# VETERİNER DOĞUM TAHMİN SİSTEMİ - GOOGLE COLAB - FINAL v5.0
# ============================================================================
# Versiyon: 5.0.0 | Biyolojik simülasyon | RF + XGBoost + MLP + LSTM + Ensemble
# Düzeltmeler: %70/15/15 bölme, SMOTE Pipeline, validation eşik, leakage yok
# ============================================================================

try:
    import google.colab; IN_COLAB=True; print("✅ Google Colab tespit edildi")
except: IN_COLAB=False; print("⚠️ Colab dışında")

if IN_COLAB:
    import subprocess
    subprocess.run(["pip","install","imbalanced-learn","shap","-q"])
    print("✅ Kütüphaneler hazır")

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings, pickle, json, os
from datetime import datetime
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, matthews_corrcoef)
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap

warnings.filterwarnings('ignore')

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam
    LSTM_AVAILABLE=True; print("✅ TensorFlow yüklendi")
except: LSTM_AVAILABLE=False; print("⚠️ TensorFlow yok, LSTM atlanacak")

if IN_COLAB:
    from google.colab import files

plt.style.use('seaborn-v0_8-darkgrid'); sns.set_palette("husl")
if IN_COLAB: plt.rcParams['figure.figsize']=(12,8); plt.rcParams['figure.dpi']=100

def safe_display(obj):
    try: display(obj)
    except: print(obj)

print("\n"+"="*70+"\nVETERİNER DOĞUM TAHMİN SİSTEMİ v5.0\n"+"="*70)

# ============================================================================
# AŞAMA 1: HAYVAN PARAMETRELERİ
# ============================================================================
# Kaynaklar: Noakes et al. 2019, Hafez & Hafez 2013, Kargo et al. 2014, Mellado et al. 2016

ANIMAL_SPECS = {
    'Cattle': {'gestation_range':(278,285),'temp_normal':38.5,'heart_rate':(65,90),'resp_rate':(25,45),'calcium_normal':9.5,'feed_intake':(30,50),'water_intake':(50,85)},
    'Sheep':  {'gestation_range':(145,152),'temp_normal':39.0,'heart_rate':(90,115),'resp_rate':(20,40),'calcium_normal':2.0,'feed_intake':(2.0,4.0),'water_intake':(5,9)},
    'Goat':   {'gestation_range':(148,153),'temp_normal':39.0,'heart_rate':(100,130),'resp_rate':(22,42),'calcium_normal':2.5,'feed_intake':(1.5,3.5),'water_intake':(4,8)},
    'Horse':  {'gestation_range':(335,345),'temp_normal':38.0,'heart_rate':(35,50),'resp_rate':(15,30),'calcium_normal':11.0,'feed_intake':(8,14),'water_intake':(35,60)},
    'Pig':    {'gestation_range':(112,118),'temp_normal':38.5,'heart_rate':(80,100),'resp_rate':(35,60),'calcium_normal':1.8,'feed_intake':(4,8),'water_intake':(12,25)},
    'Dog':    {'gestation_range':(58,68),'temp_normal':38.5,'heart_rate':(70,100),'resp_rate':(18,40),'calcium_normal':2.3,'feed_intake':(3,5),'water_intake':(1.5,3.0)},
    'Cat':    {'gestation_range':(63,67),'temp_normal':38.5,'heart_rate':(120,150),'resp_rate':(20,35),'calcium_normal':2.2,'feed_intake':(0.8,1.5),'water_intake':(1.0,2.0)},
}

FEATURE_NAMES = [
    'species_encoded','age_years','parity','gestation_days','days_before_birth',
    'body_temp_celsius','temp_change_celsius','udder_development_score','udder_edema_present',
    'teat_enlargement_score','milk_secretion_present','vulva_swelling_score',
    'vulva_discharge_present','pelvic_relaxation_score','appetite_change_percent',
    'feed_intake_kg','water_intake_liters','activity_level_score','restlessness_score',
    'nesting_behavior_present','isolation_behavior_present','aggression_score',
    'lying_frequency_per_hour','standing_frequency_per_hour','respiration_rate_per_min',
    'heart_rate_bpm','calcium_level_mmol_l','progesterone_ng_ml','labor_stage','risk_factors'
]

# ============================================================================
# AŞAMA 2: BİYOLOJİK SİMÜLASYON
# ============================================================================
print("\n[AŞAMA 2] Biyolojik Simülasyon ile Veri Üretimi\n"+"-"*70)

SPECIES_CLINICAL = {
    'Cattle': {'gestation_mu':283,'gestation_sigma':5,'temp_drop_onset':1.5,'temp_drop_max':1.0,'temp_noise':0.25,'udder_onset':14,'vulva_onset':3,'restless_onset':2,'nesting_onset':1,'base_risk':0.10,'age_risk_peak':8,'signal_noise':0.8},
    'Sheep':  {'gestation_mu':148,'gestation_sigma':4,'temp_drop_onset':1.0,'temp_drop_max':0.8,'temp_noise':0.30,'udder_onset':10,'vulva_onset':2,'restless_onset':1,'nesting_onset':0.5,'base_risk':0.12,'age_risk_peak':6,'signal_noise':1.1},
    'Goat':   {'gestation_mu':150,'gestation_sigma':4,'temp_drop_onset':1.0,'temp_drop_max':0.9,'temp_noise':0.28,'udder_onset':10,'vulva_onset':2,'restless_onset':1,'nesting_onset':0.5,'base_risk':0.13,'age_risk_peak':6,'signal_noise':1.0},
    'Horse':  {'gestation_mu':340,'gestation_sigma':14,'temp_drop_onset':2.0,'temp_drop_max':0.7,'temp_noise':0.20,'udder_onset':21,'vulva_onset':4,'restless_onset':3,'nesting_onset':1,'base_risk':0.08,'age_risk_peak':15,'signal_noise':0.9},
    'Pig':    {'gestation_mu':115,'gestation_sigma':2,'temp_drop_onset':0.8,'temp_drop_max':1.1,'temp_noise':0.35,'udder_onset':7,'vulva_onset':1,'restless_onset':0.8,'nesting_onset':0.5,'base_risk':0.15,'age_risk_peak':5,'signal_noise':1.2},
    'Dog':    {'gestation_mu':63,'gestation_sigma':2,'temp_drop_onset':1.0,'temp_drop_max':1.2,'temp_noise':0.30,'udder_onset':7,'vulva_onset':1,'restless_onset':0.5,'nesting_onset':0.3,'base_risk':0.10,'age_risk_peak':7,'signal_noise':0.7},
    'Cat':    {'gestation_mu':65,'gestation_sigma':2,'temp_drop_onset':0.8,'temp_drop_max':1.0,'temp_noise':0.28,'udder_onset':5,'vulva_onset':1,'restless_onset':0.5,'nesting_onset':0.3,'base_risk':0.09,'age_risk_peak':7,'signal_noise':0.7},
}

def sigmoid(x): return 1/(1+np.exp(-x))

def clinical_score(dbb, spec, rng):
    n=spec['signal_noise']
    # Sıcaklık
    if dbb<=spec['temp_drop_onset']*3:
        p=max(0,1-dbb/(spec['temp_drop_onset']*3)); td=spec['temp_drop_max']*(p**1.5)+rng.normal(0,0.1)
    else: td=rng.normal(0,0.08)
    td=max(0,td); bt=ANIMAL_SPECS[spec['_name']]['temp_normal']-td+rng.normal(0,spec['temp_noise']*0.3)
    # Meme
    ou=spec['udder_onset']
    ud=int(np.clip((10*(1-dbb/ou)**0.7+rng.normal(0,n)) if dbb<ou else rng.normal(2,n),1,10))
    tt=int(np.clip(ud*0.9+rng.normal(0,n*0.5),1,10))
    mk=1 if (dbb<3 and rng.random()<0.75) else (1 if (dbb<7 and rng.random()<0.35) else 0)
    # Vulva
    ov=spec['vulva_onset']
    vu=int(np.clip((10*(1-dbb/ov)**0.6+rng.normal(0,n)) if dbb<ov else rng.normal(2,n*0.7),1,10))
    dc=1 if (dbb<2 and rng.random()<0.80) else (1 if (dbb<5 and rng.random()<0.30) else 0)
    pe=int(np.clip(vu*0.85+rng.normal(0,n*0.5),1,10))
    # Davranış
    orst=spec['restless_onset']
    rs=int(np.clip((9*(1-dbb/orst)**0.5+rng.normal(0,n*0.8)) if dbb<orst else rng.normal(2.5,n),1,10))
    ne=1 if (dbb<spec['nesting_onset'] and rng.random()<0.80) else (1 if (dbb<spec['nesting_onset']*2 and rng.random()<0.25) else 0)
    is_=1 if (rs>6 and rng.random()<0.65) else 0
    ag=int(np.clip(rs*0.7+rng.normal(0,n*0.4),1,10))
    ac=int(np.clip(10-rs+rng.normal(0,n*0.5),1,10))
    ap=rng.normal(-40,12) if dbb<3 else (rng.normal(-15,10) if dbb<7 else rng.normal(0,8))
    ly=int(np.clip(rs*0.8+rng.normal(0,1.5),1,10)); st=int(np.clip(rs*0.6+rng.normal(0,1.5),1,10))
    return {'bt':bt,'td':td,'ud':ud,'tt':tt,'mk':mk,'vu':vu,'dc':dc,'pe':pe,
            'rs':rs,'ne':ne,'is_':is_,'ag':ag,'ac':ac,'ap':ap,'ly':ly,'st':st}

def birth_prob(dbb,clin,spec,parity,age,nrisk):
    raw=(4.0*sigmoid(-(dbb-1.5)/1.2)+2.5*sigmoid((clin['td']-0.4)/0.2)
         +1.5*(clin['ud']-1)/9.0+1.5*(clin['vu']-1)/9.0
         +1.2*sigmoid((clin['rs']-5)/2.0)+0.8*clin['ne']+0.6*clin['mk']
         -0.5*nrisk+min(parity*0.15,0.6)-0.4*((age-spec['age_risk_peak'])/spec['age_risk_peak'])**2)
    return float(np.clip(sigmoid(raw-3.5),0.02,0.98))

def generate_data(n_samples=10000,seed=42):
    rng=np.random.default_rng(seed)
    sp_list=list(ANIMAL_SPECS.keys()); sp_w=[0.25,0.15,0.15,0.10,0.15,0.12,0.08]
    rows=[]
    for i in range(n_samples):
        sp=rng.choice(sp_list,p=sp_w); aspec=ANIMAL_SPECS[sp]; cspec={**SPECIES_CLINICAL[sp],'_name':sp}
        age=int(rng.integers(1,12)); parity=int(rng.integers(0,7))
        gest=int(np.clip(rng.normal(cspec['gestation_mu'],cspec['gestation_sigma']),aspec['gestation_range'][0],aspec['gestation_range'][1]))
        dbb=int(np.clip(int(rng.exponential(8)),0,30))
        br=cspec['base_risk']; br+=0.08 if parity==0 else 0; br+=0.05 if age>cspec['age_risk_peak'] else 0
        nrisk=int(rng.binomial(6,br))
        hr=int(rng.integers(aspec['heart_rate'][0],aspec['heart_rate'][1]+1))
        rr=int(rng.integers(aspec['resp_rate'][0],aspec['resp_rate'][1]+1))
        ca=float(np.clip(rng.normal(aspec['calcium_normal']-(0.3 if dbb<5 else 0),0.4),aspec['calcium_normal']*0.4,aspec['calcium_normal']*1.6))
        prog=float(np.clip(rng.normal(1.5,0.8) if dbb<2 else rng.normal(5.0,2.0) if dbb<7 else rng.normal(14.0,4.0),0.2,30.0))
        fi=float(rng.uniform(aspec['feed_intake'][0],aspec['feed_intake'][1]))
        wi=float(rng.uniform(aspec['water_intake'][0],aspec['water_intake'][1]))
        clin=clinical_score(dbb,cspec,rng); prob=birth_prob(dbb,clin,cspec,parity,age,nrisk)
        succ=int(rng.random()<prob)
        ls=3 if dbb==0 else (int(rng.choice([2,3],p=[0.6,0.4])) if dbb<1 else int(rng.choice([1,2],p=[0.7,0.3])) if dbb<3 else 1)
        rows.append({'species':sp,'age_years':age,'parity':parity,'gestation_days':gest,'days_before_birth':dbb,
            'body_temp_celsius':round(clin['bt'],3),'temp_change_celsius':round(clin['td'],3),
            'udder_development_score':clin['ud'],'udder_edema_present':int(clin['ud']>6 and rng.random()<0.5),
            'teat_enlargement_score':clin['tt'],'milk_secretion_present':clin['mk'],
            'vulva_swelling_score':clin['vu'],'vulva_discharge_present':clin['dc'],
            'pelvic_relaxation_score':clin['pe'],'appetite_change_percent':round(clin['ap'],2),
            'feed_intake_kg':round(fi,2),'water_intake_liters':round(wi,2),
            'activity_level_score':clin['ac'],'restlessness_score':clin['rs'],
            'nesting_behavior_present':clin['ne'],'isolation_behavior_present':clin['is_'],
            'aggression_score':clin['ag'],'lying_frequency_per_hour':clin['ly'],
            'standing_frequency_per_hour':clin['st'],'respiration_rate_per_min':rr,
            'heart_rate_bpm':hr,'calcium_level_mmol_l':round(ca,3),'progesterone_ng_ml':round(prog,3),
            'labor_stage':ls,'risk_factors':nrisk,'successful_birth':succ})
    df_out=pd.DataFrame(rows)
    le_enc=LabelEncoder(); df_out['species_encoded']=le_enc.fit_transform(df_out['species'])
    df_out.insert(0,'animal_id',[f"{s[:3].upper()}{i+1:04d}" for i,s in enumerate(df_out['species'])])
    pos=df_out['successful_birth'].mean()
    print(f"✅ {len(df_out):,} kayıt | Başarılı:%{pos*100:.1f} | Başarısız:%{(1-pos)*100:.1f}")
    print(f"✅ Dengesizlik: {(1-pos)/pos:.2f}:1")
    return df_out,le_enc

df,le=generate_data(10000)
df.to_csv('veterinary_birth_data.csv',index=False)
print("✅ veterinary_birth_data.csv kaydedildi")
safe_display(df.head())

# ============================================================================
# AŞAMA 3: EDA
# ============================================================================
print("\n[AŞAMA 3] Keşifsel Veri Analizi\n"+"-"*70)

fig,axes=plt.subplots(2,3,figsize=(18,10)); fig.suptitle('Veteriner Doğum Verisi — EDA',fontsize=16,fontweight='bold')
sp_c=df['species'].value_counts()
axes[0,0].bar(sp_c.index,sp_c.values,color=plt.cm.Set3(range(len(sp_c)))); axes[0,0].set_title('Tür Dağılımı',fontweight='bold'); axes[0,0].tick_params(axis='x',rotation=45); axes[0,0].grid(axis='y',alpha=0.3)
axes[0,1].pie(df['successful_birth'].value_counts(),labels=['Başarılı','Başarısız'],autopct='%1.1f%%',colors=['#90EE90','#FFB6C6'],startangle=90); axes[0,1].set_title('Doğum Başarısı',fontweight='bold')
axes[0,2].hist(df['days_before_birth'],bins=30,color='coral',edgecolor='black',alpha=0.7); axes[0,2].set_title('Doğuma Kalan Gün',fontweight='bold'); axes[0,2].grid(axis='y',alpha=0.3)
axes[1,0].hist(df['temp_change_celsius'],bins=30,color='lightblue',edgecolor='black',alpha=0.7); axes[1,0].axvline(df['temp_change_celsius'].mean(),color='red',linestyle='--',linewidth=2,label='Ort'); axes[1,0].set_title('Sıcaklık Değişimi',fontweight='bold'); axes[1,0].legend(); axes[1,0].grid(axis='y',alpha=0.3)
axes[1,1].hist(df['risk_factors'],bins=20,color='gold',edgecolor='black',alpha=0.7); axes[1,1].set_title('Risk Faktörleri',fontweight='bold'); axes[1,1].grid(axis='y',alpha=0.3)
for lbl,col,nm in [(df['successful_birth']==1,'green','Başarılı'),(df['successful_birth']==0,'red','Başarısız')]:
    sub=df[lbl]; axes[1,2].scatter(sub['progesterone_ng_ml'],sub['days_before_birth'],c=col,alpha=0.3,s=10,label=nm)
axes[1,2].set_title('Progesteron vs Kalan Gün',fontweight='bold'); axes[1,2].legend(); axes[1,2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('eda_analysis.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ EDA kaydedildi")

corr=df[df.select_dtypes(include=np.number).columns].corr()
plt.figure(figsize=(20,16)); sns.heatmap(corr,mask=np.triu(np.ones_like(corr,dtype=bool)),annot=True,fmt='.2f',cmap='coolwarm',center=0,square=True,linewidths=0.5,cbar_kws={"shrink":0.8}); plt.title('Korelasyon Matrisi',fontsize=16,fontweight='bold')
plt.tight_layout(); plt.savefig('correlation_matrix.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ Korelasyon matrisi kaydedildi")

# ============================================================================
# AŞAMA 4-5: VERİ BÖLME VE STANDARDIZASYON
# ============================================================================
print("\n[AŞAMA 4-5] Veri Bölme ve Standardizasyon")

X=df[FEATURE_NAMES].copy(); y=df['successful_birth'].copy()
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=0.30,random_state=42,stratify=y)
X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=0.50,random_state=42,stratify=y_temp)
print(f"✅ Eğitim:{len(X_train):,} | Doğrulama:{len(X_val):,} | Test:{len(X_test):,}")

scaler=StandardScaler(); X_train_sc=scaler.fit_transform(X_train); X_val_sc=scaler.transform(X_val); X_test_sc=scaler.transform(X_test)
print(f"✅ Standardizasyon tamamlandı")

# ============================================================================
# AŞAMA 6: SMOTE
# ============================================================================
print("\n[AŞAMA 6] SMOTE")
print(f"Önce — 0:{(y_train==0).sum()} | 1:{(y_train==1).sum()}")
smote_obj=SMOTE(random_state=42,k_neighbors=5); X_train_bal,y_train_bal=smote_obj.fit_resample(X_train_sc,y_train)
print(f"Sonra — 0:{(y_train_bal==0).sum()} | 1:{(y_train_bal==1).sum()}")

fig,axes=plt.subplots(1,2,figsize=(12,5)); fig.suptitle('SMOTE Öncesi / Sonrası',fontweight='bold')
for ax,cnt,title in zip(axes,[y_train.value_counts(),pd.Series(y_train_bal).value_counts()],['SMOTE Öncesi','SMOTE Sonrası']):
    ax.bar(['Başarısız','Başarılı'],[cnt.get(0,0),cnt.get(1,0)],color=['#FF9999','#99CC99'],edgecolor='black')
    ax.set_title(title,fontweight='bold'); ax.set_ylabel('Örnek Sayısı')
    for j,v in enumerate([cnt.get(0,0),cnt.get(1,0)]): ax.text(j,v+20,str(v),ha='center',fontweight='bold')
    ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('smote_comparison.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ SMOTE grafiği kaydedildi")

# ============================================================================
# AŞAMA 7: HİPERPARAMETRE OPTİMİZASYONU
# ============================================================================
print("\n[AŞAMA 7] Hiperparametre Optimizasyonu\n"+"="*70)

cv5=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

print("\n🔍 Random Forest...")
rf_pipe=ImbPipeline([('smote',SMOTE(random_state=42,k_neighbors=5)),('model',RandomForestClassifier(random_state=42,n_jobs=-1))])
rf_search=RandomizedSearchCV(rf_pipe,{'model__n_estimators':[100,200,300],'model__max_depth':[8,10,15],'model__min_samples_split':[10,20,30],'model__min_samples_leaf':[5,10,20],'model__max_features':['sqrt','log2']},n_iter=15,cv=cv5,scoring='f1',random_state=42,n_jobs=-1,verbose=0)
rf_search.fit(X_train_sc,y_train); print(f"✅ RF CV F1: {rf_search.best_score_:.4f}")

print("\n🔍 XGBoost...")
xgb_pipe=ImbPipeline([('smote',SMOTE(random_state=42,k_neighbors=5)),('model',xgb.XGBClassifier(random_state=42,eval_metric='logloss',n_jobs=-1))])
xgb_search=RandomizedSearchCV(xgb_pipe,{'model__n_estimators':[100,200,300],'model__max_depth':[4,5,6],'model__learning_rate':[0.01,0.05,0.1],'model__subsample':[0.7,0.8,0.9],'model__colsample_bytree':[0.7,0.8,0.9],'model__reg_alpha':[0.1,0.5,1.0],'model__reg_lambda':[1.0,2.0,5.0],'model__gamma':[0.1,0.2,0.5]},n_iter=15,cv=cv5,scoring='f1',random_state=42,n_jobs=-1,verbose=0)
xgb_search.fit(X_train_sc,y_train); print(f"✅ XGB CV F1: {xgb_search.best_score_:.4f}")

# ============================================================================
# AŞAMA 8: MODEL EĞİTİMİ
# ============================================================================
print("\n[AŞAMA 8] Model Eğitimi\n"+"="*70)

results={}

def train_eval(name,model,Xtr,ytr,Xte,yte,Xv,yv):
    model.fit(Xtr,ytr)
    pt=model.predict_proba(Xte)[:,1]; pv=model.predict_proba(Xv)[:,1]
    # Eşik: validation seti
    pr,rc,tv=precision_recall_curve(yv,pv); f1v=np.where((pr+rc)==0,0,2*pr*rc/(pr+rc))
    bi=f1v.argmax(); thr=float(tv[bi]) if bi<len(tv) else 0.5; thr=max(0.3,min(0.7,thr))
    pred=(pt>=thr).astype(int); cm=confusion_matrix(yte,pred)
    m={'accuracy':accuracy_score(yte,pred),'balanced_accuracy':balanced_accuracy_score(yte,pred),
       'precision':precision_score(yte,pred,zero_division=0),'recall':recall_score(yte,pred,zero_division=0),
       'f1':f1_score(yte,pred,zero_division=0),'auc_roc':roc_auc_score(yte,pt),
       'auc_pr':average_precision_score(yte,pt),'mcc':matthews_corrcoef(yte,pred),
       'val_auc':roc_auc_score(yv,pv),'threshold':thr,'model':model,'predictions':pt,'val_predictions':pv}
    print(f"\n✅ {name} (eşik={thr:.2f})")
    for k,v in [('Accuracy',m['accuracy']),('Balanced Accuracy',m['balanced_accuracy']),
                ('Precision',m['precision']),('Recall',m['recall']),
                ('F1-Score',m['f1']),('AUC-ROC',m['auc_roc']),
                ('AUC-PR',m['auc_pr']),('MCC',m['mcc']),('Val AUC',m['val_auc'])]:
        print(f"   {k:<22}: {v:.4f}")
    print(f"   Başarısız doğru   : {cm[0][0]}/{(yte==0).sum()}")
    print(f"   Başarılı doğru    : {cm[1][1]}/{(yte==1).sum()}")
    return m

rf_bp={k.replace('model__',''):v for k,v in rf_search.best_params_.items()}
xgb_bp={k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}

print("\n[MODEL 1] Random Forest")
rf_model=RandomForestClassifier(**rf_bp,random_state=42,n_jobs=-1)
results['Random Forest']=train_eval('Random Forest',rf_model,X_train_bal,y_train_bal,X_test_sc,y_test,X_val_sc,y_val)

print("\n[MODEL 2] XGBoost")
xgb_model=xgb.XGBClassifier(**xgb_bp,random_state=42,eval_metric='logloss',n_jobs=-1)
results['XGBoost']=train_eval('XGBoost',xgb_model,X_train_bal,y_train_bal,X_test_sc,y_test,X_val_sc,y_val)

print("\n[MODEL 3] Neural Network (MLP)")
mlp_model=MLPClassifier(hidden_layer_sizes=(128,64,32,16),activation='relu',solver='adam',alpha=0.01,batch_size=64,learning_rate='adaptive',learning_rate_init=0.001,max_iter=1000,random_state=42,early_stopping=True,validation_fraction=0.15,n_iter_no_change=20,verbose=False)
results['Neural Network']=train_eval('Neural Network',mlp_model,X_train_bal,y_train_bal,X_test_sc,y_test,X_val_sc,y_val)

if LSTM_AVAILABLE:
    print("\n[MODEL 4] LSTM\n"+"-"*50)
    X_tr_l=X_train_bal.reshape(X_train_bal.shape[0],1,X_train_bal.shape[1])
    X_v_l=X_val_sc.reshape(X_val_sc.shape[0],1,X_val_sc.shape[1])
    X_te_l=X_test_sc.reshape(X_test_sc.shape[0],1,X_test_sc.shape[1])
    lstm_model=Sequential([LSTM(128,input_shape=(1,X_train_bal.shape[1]),return_sequences=True),Dropout(0.3),LSTM(64,return_sequences=True),Dropout(0.3),LSTM(32,return_sequences=False),Dropout(0.2),Dense(16,activation='relu'),Dropout(0.2),Dense(1,activation='sigmoid')])
    lstm_model.compile(optimizer=Adam(0.001),loss='binary_crossentropy',metrics=['accuracy',tf.keras.metrics.AUC(name='auc')])
    lstm_model.summary()
    history=lstm_model.fit(X_tr_l,y_train_bal,validation_data=(X_v_l,y_val),epochs=100,batch_size=64,
        callbacks=[EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True,verbose=1),
                   ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7,min_lr=1e-7,verbose=1)],verbose=1)
    lv_p=lstm_model.predict(X_v_l,verbose=0).flatten()
    pr,rc,tv=precision_recall_curve(y_val,lv_p); f1v=np.where((pr+rc)==0,0,2*pr*rc/(pr+rc))
    bi=f1v.argmax(); lt=float(tv[bi]) if bi<len(tv) else 0.5; lt=max(0.3,min(0.7,lt))
    lte_p=lstm_model.predict(X_te_l,verbose=0).flatten(); lte_pred=(lte_p>=lt).astype(int)
    cm_l=confusion_matrix(y_test,lte_pred)
    results['LSTM']={'accuracy':accuracy_score(y_test,lte_pred),'balanced_accuracy':balanced_accuracy_score(y_test,lte_pred),
        'precision':precision_score(y_test,lte_pred,zero_division=0),'recall':recall_score(y_test,lte_pred,zero_division=0),
        'f1':f1_score(y_test,lte_pred,zero_division=0),'auc_roc':roc_auc_score(y_test,lte_p),
        'auc_pr':average_precision_score(y_test,lte_p),'mcc':matthews_corrcoef(y_test,lte_pred),
        'val_auc':roc_auc_score(y_val,lv_p),'threshold':lt,'model':lstm_model,
        'predictions':lte_p,'val_predictions':lv_p,'history':history}
    r=results['LSTM']
    print(f"\n✅ LSTM (eşik={lt:.2f}) | F1:{r['f1']:.4f} | Bal.Acc:{r['balanced_accuracy']:.4f} | AUC:{r['auc_roc']:.4f} | MCC:{r['mcc']:.4f}")
    print(f"   Başarısız doğru: {cm_l[0][0]}/{(y_test==0).sum()} | Başarılı doğru: {cm_l[1][1]}/{(y_test==1).sum()}")
    fig,axes=plt.subplots(1,3,figsize=(18,5)); fig.suptitle('LSTM Eğitim Geçmişi',fontsize=16,fontweight='bold')
    for ax,mk,t in zip(axes,['loss','accuracy','auc'],['Kayıp','Doğruluk','AUC']):
        ax.plot(history.history[mk],label='Eğitim',linewidth=2); ax.plot(history.history[f'val_{mk}'],label='Doğrulama',linewidth=2)
        ax.set_title(t,fontweight='bold'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig('lstm_training.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ LSTM eğitim grafiği kaydedildi")

# ============================================================================
# AŞAMA 9: 5-FOLD ÇAPRAZ DOĞRULAMA
# ============================================================================
print("\n[AŞAMA 9] 5-Fold Çapraz Doğrulama\n"+"="*70)

rf_cp={k.replace('model__',''):v for k,v in rf_search.best_params_.items()}
xgb_cp={k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}
cv_pipes={
    'Random Forest':  ImbPipeline([('smote',SMOTE(random_state=42)),('model',RandomForestClassifier(**rf_cp,random_state=42,n_jobs=-1))]),
    'XGBoost':        ImbPipeline([('smote',SMOTE(random_state=42)),('model',xgb.XGBClassifier(**xgb_cp,random_state=42,eval_metric='logloss',n_jobs=-1))]),
    'Neural Network': ImbPipeline([('smote',SMOTE(random_state=42)),('model',MLPClassifier(hidden_layer_sizes=(128,64,32,16),activation='relu',solver='adam',alpha=0.01,max_iter=500,random_state=42,verbose=False))]),
}
cv_scores={}
print(f"\n{'Model':<20} {'F1 Ort':>10} {'F1 Std':>10} {'AUC Ort':>10} {'AUC Std':>10}\n"+"-"*55)
for nm,pipe in cv_pipes.items():
    f1s=cross_val_score(pipe,X_train_sc,y_train,cv=cv5,scoring='f1',n_jobs=-1)
    aucs=cross_val_score(pipe,X_train_sc,y_train,cv=cv5,scoring='roc_auc',n_jobs=-1)
    cv_scores[nm]={'f1':f1s,'auc':aucs}
    print(f"{nm:<20} {f1s.mean():>10.4f} {f1s.std():>10.4f} {aucs.mean():>10.4f} {aucs.std():>10.4f}")

nms=list(cv_scores.keys()); x=np.arange(len(nms))
fig,axes=plt.subplots(1,2,figsize=(14,5)); fig.suptitle('5-Fold Çapraz Doğrulama',fontsize=14,fontweight='bold')
for ax,metric,title in zip(axes,['f1','auc'],['F1-Score','AUC-ROC']):
    means=[cv_scores[n][metric].mean() for n in nms]; stds=[cv_scores[n][metric].std() for n in nms]
    ax.bar(x,means,yerr=stds,capsize=6,color=['#4CAF50','#2196F3','#FF9800'],edgecolor='black',alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(nms,rotation=15); ax.set_title(f'{title} (Ort ± Std)',fontweight='bold'); ax.grid(axis='y',alpha=0.3)
    for i,(m,s) in enumerate(zip(means,stds)): ax.text(i,m+s+0.005,f'{m:.3f}',ha='center',fontsize=9,fontweight='bold')
plt.tight_layout(); plt.savefig('cross_validation.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ Çapraz doğrulama kaydedildi")

# ============================================================================
# AŞAMA 10: ÖĞRENME EĞRİSİ
# ============================================================================
print("\n[AŞAMA 10] Öğrenme Eğrisi\n"+"="*70)

fig,axes=plt.subplots(1,2,figsize=(16,6)); fig.suptitle('Öğrenme Eğrileri',fontsize=16,fontweight='bold')
for ax,(nm,pipe) in zip(axes,list(cv_pipes.items())[:2]):
    szs,trs,vls=learning_curve(pipe,X_train_sc,y_train,train_sizes=np.linspace(0.1,1.0,8),cv=cv5,scoring='f1',n_jobs=-1)
    trm,trs2=trs.mean(1),trs.std(1); vlm,vls2=vls.mean(1),vls.std(1)
    ax.plot(szs,trm,'o-',color='#2196F3',label='Eğitim',linewidth=2); ax.fill_between(szs,trm-trs2,trm+trs2,alpha=0.15,color='#2196F3')
    ax.plot(szs,vlm,'o-',color='#FF5722',label='Doğrulama',linewidth=2); ax.fill_between(szs,vlm-vls2,vlm+vls2,alpha=0.15,color='#FF5722')
    ax.set_title(nm,fontweight='bold'); ax.set_xlabel('Eğitim Örnek Sayısı'); ax.set_ylabel('F1-Score'); ax.legend(); ax.grid(alpha=0.3); ax.set_ylim([0.0,1.05])
    ax.text(0.05,0.05,f"Durum: {'İyi genelleme' if vlm[-1]>0.75 else 'Yüksek yanlılık'}",transform=ax.transAxes,fontsize=9,bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.5))
plt.tight_layout(); plt.savefig('learning_curves.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ Öğrenme eğrisi kaydedildi")

# ============================================================================
# AŞAMA 11: CONFUSION MATRIX, ROC VE PR EĞRİLERİ
# ============================================================================
print("\n[AŞAMA 11] Detaylı Model Analizi\n"+"="*70)

nm=len(results); fig,axes=plt.subplots(3,nm,figsize=(6*nm,18))
if nm==1: axes=axes.reshape(-1,1)
fig.suptitle('Confusion Matrix, ROC ve PR Eğrileri',fontsize=16,fontweight='bold')
for idx,(name,m) in enumerate(results.items()):
    thr=m['threshold']; prob=m['predictions']; pred=(prob>=thr).astype(int); cm=confusion_matrix(y_test,pred)
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=axes[0,idx],annot_kws={'size':14,'weight':'bold'})
    axes[0,idx].set_title(f'{name}\nConfusion Matrix (eşik={thr:.2f})',fontweight='bold')
    axes[0,idx].set_ylabel('Gerçek'); axes[0,idx].set_xlabel('Tahmin')
    axes[0,idx].set_xticklabels(['Başarısız','Başarılı']); axes[0,idx].set_yticklabels(['Başarısız','Başarılı'])
    fpr,tpr,_=roc_curve(y_test,prob); av=roc_auc_score(y_test,prob)
    axes[1,idx].plot(fpr,tpr,linewidth=3,label=f'AUC={av:.4f}',color='darkblue'); axes[1,idx].plot([0,1],[0,1],'k--',linewidth=2,alpha=0.5); axes[1,idx].fill_between(fpr,tpr,alpha=0.15,color='blue')
    axes[1,idx].set_title(f'{name}\nROC Eğrisi',fontweight='bold'); axes[1,idx].set_xlabel('FPR'); axes[1,idx].set_ylabel('TPR'); axes[1,idx].legend(); axes[1,idx].grid(alpha=0.3)
    pa,ra,_=precision_recall_curve(y_test,prob); apv=average_precision_score(y_test,prob)
    axes[2,idx].plot(ra,pa,linewidth=3,label=f'AP={apv:.4f}',color='darkgreen'); axes[2,idx].fill_between(ra,pa,alpha=0.15,color='green'); axes[2,idx].axhline(y=y_test.mean(),color='red',linestyle='--',alpha=0.5,label=f'Baseline={y_test.mean():.2f}')
    axes[2,idx].set_title(f'{name}\nPrecision-Recall',fontweight='bold'); axes[2,idx].set_xlabel('Recall'); axes[2,idx].set_ylabel('Precision'); axes[2,idx].legend(); axes[2,idx].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('detailed_analysis.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ Detaylı analiz kaydedildi")

for name,m in results.items():
    pred=(m['predictions']>=m['threshold']).astype(int)
    print(f"\n{'='*50}\n{name}:"); print(classification_report(y_test,pred,target_names=['Başarısız','Başarılı'],digits=4))

# AŞAMA 12: Olasılık dağılımı
fig,axes=plt.subplots(1,min(4,len(results)),figsize=(6*min(4,len(results)),6))
if len(results)==1: axes=[axes]
fig.suptitle('Model Olasılık Dağılımı',fontsize=16,fontweight='bold')
for ax,(name,m) in zip(axes,results.items()):
    prob=m['predictions']
    ax.hist(prob[y_test==1],bins=30,alpha=0.7,color='green',label='Başarılı',density=True)
    ax.hist(prob[y_test==0],bins=30,alpha=0.7,color='red',label='Başarısız',density=True)
    ax.axvline(x=m['threshold'],color='black',linestyle='--',linewidth=2,label=f"Eşik={m['threshold']:.2f}")
    ax.set_title(name,fontweight='bold'); ax.set_xlabel('Tahmin Olasılığı'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('probability_distribution.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ Olasılık dağılımı kaydedildi")

# ============================================================================
# AŞAMA 13: ÖZELLİK ÖNEMİ VE SHAP
# ============================================================================
print("\n[AŞAMA 13] Özellik Önemi ve SHAP\n"+"="*70)

rf_imp=pd.DataFrame({'feature':FEATURE_NAMES,'importance':rf_model.feature_importances_}).sort_values('importance',ascending=False)
xgb_imp=pd.DataFrame({'feature':FEATURE_NAMES,'importance':xgb_model.feature_importances_}).sort_values('importance',ascending=False)
fig,axes=plt.subplots(1,2,figsize=(20,10)); fig.suptitle('Özellik Önem Analizi',fontsize=16,fontweight='bold')
for ax,imp,title,color in zip(axes,[rf_imp,xgb_imp],['Random Forest — Top 20','XGBoost — Top 20'],['teal','coral']):
    top=imp.head(20); ax.barh(range(len(top)),top['importance'].values,color=color,edgecolor='black')
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top['feature'].values); ax.set_xlabel('Önem Skoru'); ax.set_title(title,fontweight='bold'); ax.invert_yaxis(); ax.grid(axis='x',alpha=0.3)
plt.tight_layout(); plt.savefig('feature_importance.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ Özellik önemi kaydedildi")

print("\n🔍 SHAP analizi...")
idx_s=np.random.choice(len(X_test_sc),min(200,len(X_test_sc)),replace=False); shap_s=X_test_sc[idx_s]
exp=shap.TreeExplainer(rf_model); sv=exp.shap_values(shap_s); sv=sv[1] if isinstance(sv,list) else sv
plt.figure(figsize=(12,8)); shap.summary_plot(sv,shap_s,feature_names=FEATURE_NAMES,show=False,max_display=20)
plt.title('SHAP Özet Grafiği — Random Forest',fontweight='bold',pad=15)
plt.tight_layout(); plt.savefig('shap_summary.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ SHAP kaydedildi")

# ============================================================================
# AŞAMA 14: ENSEMBLE
# ============================================================================
print("\n[AŞAMA 14] Ensemble Model\n"+"="*70)

ens_raw=np.zeros(len(y_test)); ens_v=np.zeros(len(y_val)); w_list=[]
for name,m in results.items():
    w=m['f1']; w_list.append(w); ens_raw+=m['predictions']*w; ens_v+=m['val_predictions']*w
ens_raw/=sum(w_list); ens_v/=sum(w_list)
pr,rc,tv=precision_recall_curve(y_val,ens_v); f1v=np.where((pr+rc)==0,0,2*pr*rc/(pr+rc))
bi=f1v.argmax(); ens_thr=float(tv[bi]) if bi<len(tv) else 0.5; ens_thr=max(0.3,min(0.7,ens_thr))
ens_pred=(ens_raw>=ens_thr).astype(int)
ens_m={'accuracy':accuracy_score(y_test,ens_pred),'balanced_accuracy':balanced_accuracy_score(y_test,ens_pred),
       'precision':precision_score(y_test,ens_pred,zero_division=0),'recall':recall_score(y_test,ens_pred,zero_division=0),
       'f1':f1_score(y_test,ens_pred,zero_division=0),'auc_roc':roc_auc_score(y_test,ens_raw),
       'auc_pr':average_precision_score(y_test,ens_raw),'mcc':matthews_corrcoef(y_test,ens_pred)}
print(f"\n📊 Ensemble (eşik={ens_thr:.3f})")
for k,v in ens_m.items(): print(f"   {k:<22}: {v:.4f}")
print("\nModel ağırlıkları:")
for name,w in zip(results.keys(),w_list): print(f"   {name:<20}: {w/sum(w_list):.3f} (%{w/sum(w_list)*100:.1f})")

# ============================================================================
# AŞAMA 15: BOOTSTRAP
# ============================================================================
print("\n[AŞAMA 15] Bootstrap %95 Güven Aralığı\n"+"="*70)

def bootstrap_ci(y_true,proba,thr=0.5,n=1000):
    scores=[]; y=np.array(y_true)
    for _ in range(n):
        idx=np.random.choice(len(y),len(y),replace=True)
        scores.append(f1_score(y[idx],(proba[idx]>=thr).astype(int),zero_division=0))
    return np.percentile(scores,2.5),np.percentile(scores,97.5)

print(f"\n{'Model':<20} {'F1':>8} {'%95 Alt':>10} {'%95 Üst':>10}\n"+"-"*52)
all_names=[]; all_f1=[]; all_lo=[]; all_hi=[]
for name,m in results.items():
    lo,hi=bootstrap_ci(y_test,m['predictions'],m['threshold'])
    print(f"{name:<20} {m['f1']:>8.4f} {lo:>10.4f} {hi:>10.4f}")
    all_names.append(name); all_f1.append(m['f1']); all_lo.append(lo); all_hi.append(hi)
lo_e,hi_e=bootstrap_ci(y_test,ens_raw,ens_thr)
print(f"{'Ensemble':<20} {ens_m['f1']:>8.4f} {lo_e:>10.4f} {hi_e:>10.4f}")
all_names.append('Ensemble'); all_f1.append(ens_m['f1']); all_lo.append(lo_e); all_hi.append(hi_e)

fig,ax=plt.subplots(figsize=(10,5))
ye_low=[f-lo for f,lo in zip(all_f1,all_lo)]; ye_high=[hi-f for f,hi in zip(all_f1,all_hi)]
ax.bar(all_names,all_f1,yerr=[ye_low,ye_high],capsize=8,color='steelblue',edgecolor='black',alpha=0.8)
ax.set_title('F1-Score — %95 Bootstrap Güven Aralıkları',fontweight='bold'); ax.set_ylabel('F1-Score'); ax.grid(axis='y',alpha=0.3)
for i,v in enumerate(all_f1): ax.text(i,v+ye_high[i]+0.005,f'{v:.3f}',ha='center',fontsize=9,fontweight='bold')
plt.tight_layout(); plt.savefig('bootstrap_ci.png',dpi=150,bbox_inches='tight'); plt.show(); print("✅ Bootstrap kaydedildi")

# ============================================================================
# AŞAMA 16: ÖZET TABLO
# ============================================================================
print("\n[AŞAMA 16] Performans Özet Tablosu\n"+"="*70)

rows=[]
for name,m in results.items():
    rows.append({'Model':name,'Accuracy':f"{m['accuracy']:.4f}",'Bal.Acc':f"{m['balanced_accuracy']:.4f}",
                 'Precision':f"{m['precision']:.4f}",'Recall':f"{m['recall']:.4f}",
                 'F1-Score':f"{m['f1']:.4f}",'AUC-ROC':f"{m['auc_roc']:.4f}",
                 'AUC-PR':f"{m['auc_pr']:.4f}",'MCC':f"{m['mcc']:.4f}"})
rows.append({'Model':'Ensemble','Accuracy':f"{ens_m['accuracy']:.4f}",'Bal.Acc':f"{ens_m['balanced_accuracy']:.4f}",
             'Precision':f"{ens_m['precision']:.4f}",'Recall':f"{ens_m['recall']:.4f}",
             'F1-Score':f"{ens_m['f1']:.4f}",'AUC-ROC':f"{ens_m['auc_roc']:.4f}",
             'AUC-PR':f"{ens_m['auc_pr']:.4f}",'MCC':f"{ens_m['mcc']:.4f}"})
summary_df=pd.DataFrame(rows)
print("\n"+"="*110); print(summary_df.to_string(index=False)); print("="*110)

fig,ax=plt.subplots(figsize=(22,len(rows)*0.8+2)); ax.axis('tight'); ax.axis('off')
tbl=ax.table(cellText=summary_df.values,colLabels=summary_df.columns,cellLoc='center',loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1,2.2)
for i in range(len(summary_df.columns)): tbl[(0,i)].set_facecolor('#1565C0'); tbl[(0,i)].set_text_props(weight='bold',color='white')
for col in ['Accuracy','Bal.Acc','Precision','Recall','F1-Score','AUC-ROC','AUC-PR','MCC']:
    ci=summary_df.columns.get_loc(col); vals=pd.to_numeric(summary_df[col],errors='coerce'); mx,mn=vals.max(),vals.min()
    for ri in range(len(summary_df)):
        cell=tbl[(ri+1,ci)]; v=vals.iloc[ri]
        if v==mx: cell.set_facecolor('#C8E6C9'); cell.set_text_props(weight='bold',color='#1B5E20')
        elif v==mn: cell.set_facecolor('#FFECB3'); cell.set_text_props(color='#E65100')
        else: cell.set_facecolor('#F5F5F5')
for ri in range(len(summary_df)): tbl[(ri+1,0)].set_facecolor('#EEEEEE'); tbl[(ri+1,0)].set_text_props(weight='bold')
for ci in range(len(summary_df.columns)): tbl[(len(summary_df),ci)].set_facecolor('#E3F2FD')
plt.title('Model Performans Özet Tablosu\n',fontsize=14,fontweight='bold')
plt.savefig('model_performance_summary_table.png',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print("✅ Özet tablo kaydedildi")

# ============================================================================
# AŞAMA 17: MODELLERİ KAYDET
# ============================================================================
print("\n[AŞAMA 17] Modelleri Kaydetme\n"+"="*70)

os.makedirs('saved_models',exist_ok=True)
for obj,fname in [(scaler,'scaler.pkl'),(le,'label_encoder.pkl'),(rf_model,'rf_model.pkl'),(xgb_model,'xgb_model.pkl'),(mlp_model,'mlp_model.pkl')]:
    with open(f'saved_models/{fname}','wb') as f: pickle.dump(obj,f); print(f"✅ {fname}")
if LSTM_AVAILABLE and 'LSTM' in results:
    results['LSTM']['model'].save('saved_models/lstm_model.h5'); print("✅ lstm_model.h5")
with open('saved_models/metadata.json','w') as f:
    json.dump({'feature_columns':FEATURE_NAMES,'split_ratio':'70/15/15','cv_folds':5,'smote_applied':True,
               'smote_in_pipeline':True,'threshold_from':'validation_set','data_generation':'biological_simulation',
               'rf_best_params':rf_search.best_params_,'xgb_best_params':xgb_search.best_params_,
               'optimal_thresholds':{n:float(m['threshold']) for n,m in results.items()},
               'ensemble_threshold':float(ens_thr),'training_date':datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
               'n_samples':len(df),'n_features':len(FEATURE_NAMES),'version':'5.0'},f,indent=4)
print("✅ metadata.json")

if IN_COLAB:
    print("\n"+"="*70+"\nİNDİRME:\n  !zip -r saved_models.zip saved_models/\n  files.download('saved_models.zip')")

best=max(results,key=lambda n:results[n]['balanced_accuracy'])
lstm_line="   • LSTM\n" if 'LSTM' in results else ""
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║     VETERİNER DOĞUM TAHMİN SİSTEMİ v5.0 — TAMAMLANDI ✅        ║
╚══════════════════════════════════════════════════════════════════╝
📊 {len(df):,} kayıt | {df['species'].nunique()} tür | Bölme: %70/15/15 | SMOTE: Pipeline
🤖 RF • XGBoost • MLP
{lstm_line}   • Ensemble
🏆 EN İYİ: {best} | Bal.Acc:{results[best]['balanced_accuracy']:.4f} | F1:{results[best]['f1']:.4f} | AUC:{results[best]['auc_roc']:.4f}
📊 ENSEMBLE: Bal.Acc:{ens_m['balanced_accuracy']:.4f} | F1:{ens_m['f1']:.4f} | AUC:{ens_m['auc_roc']:.4f}
Tarih: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
""")

In [ ]:
!zip -r saved_models.zip saved_models/
files.download('saved_models.zip')